In [1]:
import pandas as pd
pd.options.mode.chained_assignment = None
import numpy as np
from openpyxl import load_workbook
from datetime import datetime
import re

In [3]:
clients = ['CLR','PT',"MA","NOK","ALC","JOD","SESA","HUW"]
col_names = ["Date","Location","Item","ExpectedQty","CountedQty","Variance"]

def loadDataLog():
    df = pd.read_excel("DATA_LOG.xlsx", sheet_name="SUMMARY", skiprows=4)

    CLR = df.iloc[:,0:6]
    PT = df.iloc[:,7:13]
    MA = df.iloc[:,14:20]
    NOK = df.iloc[:,21:27]
    ALC = df.iloc[:,28:34]
    JOD = df.iloc[:,35:41]
    SESA = df.iloc[:,42:48]
    HUW = df.iloc[:,49:55]
    
    ALL = [CLR,PT,MA,NOK,ALC,JOD,SESA,HUW]
    
    return ALL

ALL = loadDataLog()


def concatDataLog(ALL,col_names,clients):
    a = 0
    for cl in ALL:
        cl.columns = col_names
        cl['CLIENT'] = clients[a]
        a = a+1
        cl.dropna(subset="Date", inplace=True)
        cl.dropna(subset="Location", inplace=True)
        cl.dropna(subset="ExpectedQty", inplace=True)
        
    countData = pd.DataFrame(columns = ["Date","Location","Item","ExpectedQty","CountedQty","Variance","CLIENT"])
    
    for cl  in ALL:
        countData = pd.concat([countData,cl],ignore_index=True)
    
    
    countData['Item'] = countData['CLIENT'].astype(str)+"_"+countData['Item'].astype(str)
    countData["Date"] = pd.to_datetime(countData["Date"], errors='coerce')
    countData["Date"] = countData["Date"].dt.strftime("%Y-%m-%d")
    countData['Month'] = countData['Date'].str[5:7].replace({
        '01': 'Jan', '02': 'Feb', '03': 'Mar', '04': 'Apr',
        '05': 'May', '06': 'Jun', '07': 'Jul', '08': 'Aug',
        '09': 'Sep', '10': 'Oct', '11': 'Nov', '12': 'Dec'
    })
    
    
    countData['Variance'] = countData['CountedQty'].astype(float)- countData['ExpectedQty'].astype(float)
    
    return countData
   
countData = concatDataLog(ALL,col_names,clients)




C:\Users\lungelo.gwala\AppData\Local\Temp\ipykernel_21824\4008189845.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  countData = pd.concat([countData,cl],ignore_index=True)


In [65]:
countData.to_excel("countData.xlsx", index=False)

In [67]:
clients = ['CLR','PT',"MA","NOK","ALC","JOD","SESA","HUW"]

CLRh = ["BIN_LOCATION","SKU","QUANTITY"]
PTh = ['Storage Bin',"Product", "Quantity"]
MAh = ["Storage Bin","Material","Available stock"]
NOKh = ['Location', 'Item code',"Quantity in stock"]
ALCh = ['Location', 'Item code',"Quantity in stock"]
JODh = ['LOCATOR','PART','QUANTITY']
SESAh = ['Storage Bin', 'Material', 'Total Stock']
HUWh = ['Locator','Item','Inv Qty.']

ALLh = [CLRh,PTh,MAh,NOKh,ALCh,JODh,SESAh,HUWh]


def loadSOH(clients,ALLh):
    cols_name = ['Location',"Item","Qty"]
    SOH = []
    a = 0
    for el in clients:
        el = pd.read_excel("SOH.xlsx",sheet_name = el, usecols = ALLh[a])
        
        if clients[a] == "CLR":
            el = el.rename(columns={"BIN_LOCATION":"Location","SKU":"Item","QUANTITY":"Qty"})
        elif clients[a] == "PT":
            el = el.rename(columns={"Storage Bin":"Location","Product":"Item","Quantity":"Qty"})
        elif clients[a] == "MA":
            el = el.rename(columns={"Storage Bin":"Location","Material":"Item","Available stock":"Qty"})
        elif clients[a] == "NOK":
            el = el.rename(columns={"Location":"Location","Item code":"Item","Quantity in stock":"Qty"})
        elif clients[a] == "ALC":
            el = el.rename(columns={"Location":"Location","Item code":"Item","Quantity in stock":"Qty"})
        elif clients[a] == "JOD":
            el = el.rename(columns={"LOCATOR":"Location","PART":"Item","QUANTITY":"Qty"})
        elif clients[a] == "SESA":
            el = el.rename(columns={"Storage Bin":"Location","Material":"Item","Total Stock":"Qty"})
        elif clients[a] == "HUW":
            el = el.rename(columns={"Locator":"Location","Item":"Item","Inv Qty.":"Qty"})
            
        
        el['Item'] = el['Item'].astype(str)
        if clients[a] == "NOK" or clients[a] == "ALC":
            el['Item'] = el['Item'].str[4:]
 
        
        
        el['Client'] = clients[a]
        
        el['Item'] = el['Client'].astype(str)+"_"+el['Item'].astype(str)
        
        el['KEY'] = el['Location'].astype(str)+"_"+el['Item'].astype(str)
        SOH.append(el)
        
        
        a = a +1
    return SOH

SOH = loadSOH(clients,ALLh)

def concatSOH(SOH):
    newSOH = pd.DataFrame(columns = ["KEY","Location","Item","Qty","Client"])
    for el in SOH:
        newSOH = pd.concat([newSOH,el],ignore_index=True, sort=False)

    
    
    final = newSOH.groupby(['KEY',"Location","Item","Client"])['Qty'].sum()
    final = pd.DataFrame(final)
    final.reset_index(inplace=True)
    
    return final

final = concatSOH(SOH)

In [69]:
final.to_excel("FinalSOH.xlsx", index=False)

In [81]:
def setup(final,countData):
    setup = pd.read_excel("PomonaCounts_Summary.xlsx",sheet_name="SETUP",usecols = ['Client','Item','ABC'])
    con = pd.read_excel("PomonaCounts_Summary.xlsx",sheet_name="CONDITION",usecols = ["KEYY","FREQ"])
    
    instock = final[["Item","Client"]].drop_duplicates(subset="Item")
    newcount = countData[['Item',"CLIENT"]].drop_duplicates(subset="Item")
    newcount.rename(columns={"CLIENT":"Client"},inplace=True)
    
    def abc(row):
        if row['Client'] == "PT" or row['Client'] == "MA" or row['Client'] == "SESA":
            return "D"
        else:
            return "X"
    instock['ABC'] = instock.apply(lambda row : abc(row), axis =1)
    newcount['ABC'] = newcount.apply(lambda row : abc(row), axis =1)
    
    
    newList = instock[~instock['Item'].isin(setup['Item'])]
    newListn = newcount[~newcount['Item'].isin(setup['Item'])]
    
    setup = pd.concat([setup,newList],ignore_index=True)
    setup = pd.concat([setup,newListn],ignore_index=True)
    
    setup['KEYY'] = setup['Client'].astype(str)+"_"+setup['ABC'].astype(str)
    setup = pd.merge(setup,con, on='KEYY',how='left')
    setup['FREQ'] = setup['FREQ'].astype(int)
    
    return setup,instock
setup,instock = setup(final,countData)

C:\Users\lungelo.gwala\AppData\Local\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:81: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():
C:\Users\lungelo.gwala\AppData\Local\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:81: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():


In [83]:
def targets(setup,final):
    noBins = final.groupby(['Item'])['Location'].count()
    noBins = pd.DataFrame(noBins)
    noBins.reset_index(inplace=True)
    setup = pd.merge(setup, noBins, on='Item',how='left')
    setup['Location'] = setup['Location'].fillna(0)
    setup['Location'] = setup['Location'].astype(int)
    setup['BinTarget'] = setup['FREQ']*setup['Location']
    
    return setup
setup = targets(setup,final)

In [85]:
def progress(setup,countData,instock):
    
    instock_items = set(instock['Item'])
    setup['InStock'] = np.where(setup['Item'].isin(instock_items), 'YES', 'NO')
    
    occ = countData.groupby(['Item'])['Location'].count()
    occ = pd.DataFrame(occ)
    occ.reset_index(inplace=True)
    occ = occ.rename(columns ={"Location":"TotalCounts"})

    unq = countData.drop_duplicates(subset = ["Location","Item"])
    unq = unq.groupby(['Item'])['Location'].count()
    unq = pd.DataFrame(unq)
    unq.reset_index(inplace=True)
    unq = unq.rename(columns ={"Location":"UniqueBinCount"})
    
    setup = pd.merge(setup,occ, on='Item', how='left')
    setup = pd.merge(setup,unq, on='Item', how='left')
    
    setup['TotalCounts'] = setup['TotalCounts'].fillna(0)
    setup['UniqueBinCount'] = setup['UniqueBinCount'].fillna(0)
    
    return setup

setup = progress(setup,countData,instock)
    

In [87]:
setup.to_excel("setup.xlsx",index=False)

In [3]:
df = pd.read_excel("TEST.xlsx")
df['TRIM'] = df['TRIM'].apply(lambda x: re.sub(r'\s+', '', x))
def chunk_string(string, chunk_size=10):
    return [string[i:i+chunk_size] for i in range(0, len(string), chunk_size)]

chunks_list = []

# Iterate over each row in the original dataframe, split strings into chunks, and append to the list
for index, row in df.iterrows():
    chunks = chunk_string(row['TRIM'])
    chunks_list.extend(chunks)

# Create a new dataframe from the list of chunks
new_df = pd.DataFrame({'chunk': chunks_list})
new_df.to_excel("SN.xlsx")
dff = pd.read_excel("TEST.xlsx",sheet_name="ALL")
new_df.rename(columns={"chunk":"S/N"},inplace=True)
df2_exploded = dff.assign(SN=dff['S/N'].str.split(r'[\s\n]+')).explode('SN')
merged_df = new_df.merge(df2_exploded, left_on='S/N', right_on='SN', how='left')
merged_df.to_excel("check.xlsx")